In [24]:
#importing Libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [25]:
# Define the number of samples
num_samples = 5000

# Feature 1: Consistency of Work & Gigs (0-10, higher is better)
consistency_of_work = np.random.randint(1, 11, num_samples)

# Feature 2: Freelancing Experience (in years, 0-20 years)
freelancing_experience = np.random.randint(0, 21, num_samples)

# Feature 3: Income Diversification (number of income sources, 1-5)
income_diversification = np.random.randint(1, 6, num_samples)

# Feature 4: Financial Risk Events (number of platform bans, 0-3)
financial_risk_events = np.random.randint(0, 4, num_samples)

# Generate FICO Scores (300-850)
fico_scores = np.random.randint(300, 851, num_samples)

# Calculate Credit Score
credit_score = 500 * (fico_scores - 300)

In [26]:
# Assign Interest Rates based on FICO Score brackets
interest_rates = []
for fico in fico_scores:
    if 300 <= fico <= 500:
        interest_rates.append(None)  # Not eligible
    elif 501 <= fico <= 600:
        interest_rates.append(np.random.uniform(18, 22))
    elif 601 <= fico <= 700:
        interest_rates.append(np.random.uniform(15, 18))
    elif 701 <= fico <= 800:
        interest_rates.append(np.random.uniform(10, 15))
    else:
        interest_rates.append(np.random.uniform(7, 10))

In [27]:
# Calculate IScore based on weighted features
iScore = (consistency_of_work * 0.30 +
          freelancing_experience * 0.25 +
          income_diversification * 0.20 -
          financial_risk_events * 0.25 +
          (club_names / 5000000) * 0.30 +  # Normalize asset values
          (university_names / 5000) * 0.40 +
          (car_models / 5000000) * 0.30) * 100

# Normalize iScore (Scale between 300-900)
iScore = np.interp(iScore, (iScore.min(), iScore.max()), (300, 900))

In [28]:
# Generate Dummy Data for New Columns
names = [f"{i}" for i in range(num_samples)]
addresses = [f"{i}" for i in range(num_samples)]
club_names = np.random.randint(10000, 5000001, num_samples)  # Range: 10,000 to 5,000,000
university_names = np.random.randint(100, 5001, num_samples)  # Range: 100 to 5,000
car_models = np.random.randint(100000, 5000001, num_samples)  # Range: 100,000 to 5,000,000

In [29]:
# Create DataFrame
df = pd.DataFrame({
    "Name": names,
    "Address": addresses,
    "Club Name": club_names,
    "University Name": university_names,
    "Car Model": car_models,
    "Consistency of Work & Gigs": consistency_of_work,
    "Freelancing Experience (years)": freelancing_experience,
    "Income Diversification": income_diversification,
    "Financial Risk Events": financial_risk_events,
    "FICO Score": fico_scores,
    "Credit Score": credit_score,
    "IScore": iScore,
    "Interest Rate (%)": interest_rates
})

In [30]:
# Create Loan Eligibility Column based on FICO Score
df["Loan Eligibility"] = df["FICO Score"].apply(lambda x: "Eligible" if x > 500 else "Not Eligible")

# Fill missing interest rates with 0 (or choose a meaningful value)
df["Interest Rate (%)"] = df["Interest Rate (%)"].fillna(0)

# Encode categorical columns
label_encoder = LabelEncoder()
df["Loan Eligibility"] = label_encoder.fit_transform(df["Loan Eligibility"])  # "Eligible" -> 1, "Not Eligible" -> 0

In [31]:
# Save the processed DataFrame
df.to_csv("credit_data_processed.csv", index=False)

# Load dataset
data = pd.read_csv("credit_data_processed.csv")

In [32]:
# Feature selection (all columns except target)
X = data.drop(columns=["Credit Score"])
y = data["Credit Score"]

# First Split: Train (80%) and Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Second Split: Train (80% of the previous train set) and Validation (20%)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [33]:
# Train ML Models with Regularization to avoid overfitting
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=10),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
}

In [34]:
#Checking Performance of the models 
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    print(f"{name} Performance on Validation Set:")
    print(f"MAE: {mean_absolute_error(y_val, y_pred)}")
    print(f"MSE: {mean_squared_error(y_val, y_pred)}")
    print(f"R^2 Score: {r2_score(y_val, y_pred)}\n")

Linear Regression Performance on Validation Set:
MAE: 7.288690540008247e-11
MSE: 8.208543352282222e-21
R^2 Score: 1.0

Decision Tree Performance on Validation Set:
MAE: 36.25
MSE: 20000.0
R^2 Score: 0.9999967396147202

Random Forest Performance on Validation Set:
MAE: 34.8653125
MSE: 3255.266484375
R^2 Score: 0.9999994693288536

XGBoost Performance on Validation Set:
MAE: 180.12392440795898
MSE: 69141.88510881294
R^2 Score: 0.999988728540779

